# Pipeline RAG: Consultor de Continuidad Transformers

## Objetivos de Aprendizaje

- Implementar el pipeline Retrieval-Augmented Generation (RAG) completo
- Crear embeddings semánticos con `text-embedding-3-small`
- Construir un índice vectorial con FAISS para búsqueda por similitud coseno
- Integrar retrieval + LLM para análisis de consistencia narrativa

In [ ]:
!pip install openai langchain langchain-openai langchain-community faiss-cpu numpy python-dotenv

## 1. Configuración

Usamos las variables de entorno `GITHUB_BASE_URL` y `GITHUB_TOKEN` para conectar con la API.

In [ ]:
import os
import json
import time
import numpy as np
from dataclasses import dataclass
from dotenv import load_dotenv
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA
from IPython.display import display, Markdown

load_dotenv()

client = OpenAI(
    base_url=os.getenv("GITHUB_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
)

print("✓ Cliente OpenAI configurado")

## 2. Carga del Lore Canónico

Importamos los fragmentos de la base de conocimiento. Los datos integran fuentes internas (biblias de personajes, notas de producción) y externas (Transformers Wiki, IMDb), consolidadas manualmente en `lore_database.py` para el prototipo.

In [ ]:
from lore_database import get_all_fragments, get_fragments_by_category

fragments = get_all_fragments()
texts = [f.text for f in fragments]
metadatas = [{"id": f.id, "category": f.category, "film": f.film} for f in fragments]

print(f"✓ {len(fragments)} fragmentos cargados")
for cat in ["PERSONAJE", "EVENTO", "OBJETO", "TIMELINE", "FACCION"]:
    n = len(get_fragments_by_category(cat))
    print(f"  {cat}: {n}")

## 3. Embeddings Semánticos

Cada fragmento se convierte a un vector de 1536 dimensiones con `text-embedding-3-small`.

In [ ]:
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.environ.get("GITHUB_TOKEN"),
    base_url=os.getenv("GITHUB_BASE_URL"),
)

# probamos un embedding
test_emb = embeddings_model.embed_query("Bumblebee voz radio")
print(f"✓ Embedding generado: {len(test_emb)} dimensiones")
print(f"  Primeros 5 valores: {[round(v, 4) for v in test_emb[:5]]}")

## 4. Índice Vectorial FAISS

Construimos el índice vectorial con todos los fragmentos del lore.
FAISS (Facebook AI Similarity Search) permite búsqueda eficiente por similitud coseno.

In [ ]:
print("Construyendo indice FAISS...")
vectorstore = FAISS.from_texts(texts, embeddings_model, metadatas=metadatas)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print(f"✓ Indice construido con {len(texts)} vectores")

## 5. Búsqueda Semántica

Dado un fragmento de guion, recuperamos los 4 fragmentos del lore más relevantes.

In [ ]:
query = "Bumblebee habla en voz alta con Sam en 2007"
docs = retriever.invoke(query)

display(Markdown(f"**Consulta:** {query}\n\n**Fragmentos recuperados:**"))
for i, doc in enumerate(docs, 1):
    meta = doc.metadata
    print(f"[{i}] [{meta.get('id', '?')} | {meta.get('category', '?')}]")
    print(f"    {doc.page_content[:120]}...\n")

## 6. Prompts para el Consultor

Definimos tres prompts: **Zero-Shot** para validación general, **Few-Shot** para clasificar por tipo de inconsistencia, y **Chain-of-Thought** para casos con razonamiento multi-paso.

In [ ]:
SYSTEM_PROMPT = """Eres el Consultor de Continuidad del universo Transformers (saga Michael Bay, 2007-2023).
Al analizar un fragmento de guion debes:
1. Identificar personajes, eventos y objetos mencionados
2. Contrastarlos con el lore recuperado
3. Clasificar cada inconsistencia: CRITICA / MODERADA / MENOR
4. Citar la fuente especifica que contradice el guion

Reglas:
- Basa tu analisis solo en el contexto de lore proporcionado
- Si el contexto no cubre algo, indicalo en vez de inventar
- Severidades: CRITICA (rompe la trama), MODERADA (contradice hechos), MENOR (detalle)

Responde siempre en JSON puro:
{
  "verdict": "CONSISTENTE | INCONSISTENTE | REQUIERE_REVISION",
  "elements_detected": ["elemento1"],
  "inconsistencies": [
    {
      "description": "descripcion",
      "guion_says": "lo que dice el guion",
      "lore_says": "lo que dice el lore",
      "severity": "CRITICA | MODERADA | MENOR",
      "lore_source": "id del fragmento"
    }
  ],
  "recommendation": "accion para el guionista",
  "confidence_score": 0.95
}"""

FEW_SHOT_EXAMPLES = """
Guion: "Sam y Cade se conocen en la base militar"
Lore: Son protagonistas de arcos distintos (TF1-3 y TF4+)
Tipo: INCONSISTENCIA_TEMPORAL

Guion: "Bumblebee le dice a Sam: 'Confio en ti'"
Lore: Bumblebee no tiene voz hasta la pelicula Bumblebee (2018)
Tipo: INCONSISTENCIA_PERSONAJE

Guion: "El AllSpark sigue intacto en el Sector 7"
Lore: El AllSpark fue destruido al final de Transformers (2007)
Tipo: INCONSISTENCIA_OBJETO
"""

COT_TEMPLATE = """Analiza este fragmento paso a paso:

FRAGMENTO: {fragmento}
CONTEXTO RAG: {contexto}

PASO 1 - INVENTARIO: Que personajes, fechas y objetos aparecen?
PASO 2 - TEMPORALIDAD: La fecha es consistente con la saga?
PASO 3 - ESTADO DE PERSONAJES: Coincide con el canon?
PASO 4 - LOGICA INTERNA: Las motivaciones son coherentes?
PASO 5 - DICTAMEN: Hay inconsistencias? Que nivel tienen?

Razonamiento:"""

print("✓ Prompts definidos (Zero-Shot, Few-Shot, Chain-of-Thought)")

## 7. Generación con Contexto Aumentado

Los fragmentos recuperados se inyectan como contexto en el prompt antes de enviarlo al LLM.

In [ ]:
@dataclass
class ContinuityReport:
    fragment_analyzed: str
    verdict: str
    elements_detected: list
    inconsistencies: list
    recommendation: str
    retrieved_lore: list
    confidence_score: float
    processing_time_ms: float


def analyze(fragment):
    t0 = time.time()
    docs = retriever.invoke(fragment)

    context = "\n".join([
        f"[{doc.metadata.get('id', '?')} | {doc.metadata.get('category', '?')}]\n{doc.page_content}"
        for doc in docs
    ])
    user_prompt = f"CONTEXTO LORE:\n{context}\n\nGUION:\n{fragment}\n\nAnaliza y responde en JSON."

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=0.1,
        max_tokens=1000,
        response_format={"type": "json_object"},
    )

    elapsed = (time.time() - t0) * 1000
    p = json.loads(response.choices[0].message.content)

    return ContinuityReport(
        fragment_analyzed=fragment,
        verdict=p.get("verdict", "REQUIERE_REVISION"),
        elements_detected=p.get("elements_detected", []),
        inconsistencies=p.get("inconsistencies", []),
        recommendation=p.get("recommendation", ""),
        retrieved_lore=[doc.page_content[:120] + "..." for doc in docs],
        confidence_score=float(p.get("confidence_score", 0.5)),
        processing_time_ms=round(elapsed, 2),
    )

print("✓ Funcion analyze() definida")

## 8. Prueba del Pipeline Completo

In [ ]:
# TC_001 - Bumblebee habla (inconsistencia critica)
fragmento = "Escena 14. 2007. Bumblebee se gira hacia Sam y dice en voz alta: 'Sam, debes llevar el AllSpark al Secretario de Defensa.'"
report = analyze(fragmento)

display(Markdown(f"**Fragmento analizado:**\n> {fragmento[:100]}..."))
display(Markdown(f"**Veredicto:** `{report.verdict}` | **Confianza:** {report.confidence_score:.0%}"))
display(Markdown(f"**Elementos detectados:** {', '.join(report.elements_detected)}"))

if report.inconsistencies:
    print(f"\nInconsistencias ({len(report.inconsistencies)}):")
    for inc in report.inconsistencies:
        print(f"  [{inc.get('severity')}] {inc.get('description')}")
        print(f"    guion: {inc.get('guion_says')}")
        print(f"    lore:  {inc.get('lore_says')}")

print(f"\nRecomendacion: {report.recommendation}")
print(f"Tiempo: {report.processing_time_ms:.0f} ms")

In [ ]:
# TC_006 - caso consistente
fragmento2 = "Optimus Prime despliega su espada energetica y enfrenta a Megatron en las calles de Mission City. Sam corre sosteniendo el AllSpark."
report2 = analyze(fragmento2)

display(Markdown(f"**Veredicto:** `{report2.verdict}` | **Confianza:** {report2.confidence_score:.0%}"))
print(f"Recomendacion: {report2.recommendation}")
print(f"Tiempo: {report2.processing_time_ms:.0f} ms")

## Conclusiones

- El pipeline RAG encadena Retrieval (FAISS) → Augmentation (contexto en prompt) → Generation (GPT-4o)
- `temperature=0.1` entrega respuestas deterministas, necesario para validación de hechos
- El chunking atómico (1 hecho por fragmento) evita que el retrieval recupere chunks con ruido